# BASTION Paper Figures: Simulated Data (Figures 1 & 2)

Reproduces Figures 1 and 2 from the BASTION paper.

- **Figure 1**: Scatter + estimated Trend (black), Trend+Seasonality (blue), outlier positions (red dashed).
- **Figure 2**: 4-panel trend comparison TBATS / MSTL / STR / BASTION with MSE.

`fit_BASTION` with `Ks=[7,30]`, `Outlier=True`, `nsave=2000, nburn=5000` (~2–4 min).
R reference `.rds` files are loaded when available (graceful fallback if missing).

In [ ]:
import os, sys, warnings
import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_DIR  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
OUTPUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
EXTDATA_DIR  = os.path.join(PROJECT_DIR, 'BASTION', 'inst', 'extdata')
os.makedirs(OUTPUT_DIR, exist_ok=True)

sys.path.insert(0, PROJECT_DIR)
from pybastion import fit_BASTION

print(f'Project root : {PROJECT_DIR}')

In [ ]:
def generate_T(n=500):
    bp = [0, 125, 250, 375, 500]
    slopes = [0.5, -0.3, 0.8, -0.5]
    T = np.zeros(n)
    for i in range(len(slopes)):
        seg = np.arange(bp[i], bp[i + 1])
        T[seg] = T[bp[i]] + slopes[i] * np.arange(len(seg))
    return T

def gen_sim(n=500, seed=4):
    rng2 = np.random.default_rng(seed)
    t    = np.arange(1, n + 1)
    T    = generate_T(n)
    S7   = 5 * np.sin(2 * np.pi * t / 7)
    S30  = 3 * np.cos(2 * np.pi * t / 30)
    eps  = rng2.normal(0, 1, n)
    y    = T + S7 + S30 + eps
    out_idx = [50, 200, 400]
    for i in out_idx:
        y[i] += rng2.choice([-1, 1]) * rng2.uniform(15, 25)
    return {'y': y, 'T': T, 'S7': S7, 'S30': S30, 'outliers': out_idx}

sim = gen_sim(n=500, seed=4)
print(f"Simulated series: n={len(sim['y'])}, outliers at {sim['outliers']}")

In [ ]:
print('Fitting BASTION (nsave=2000, nburn=5000, nchains=2) ... (~2-4 min)')
result = fit_BASTION(
    sim['y'],
    Ks=[7, 30],
    Outlier=True,
    cl=0.95,
    obsSV='const',
    nsave=2000,
    nburn=5000,
    nchains=2,
    seed=40,
)
summary = result['summary']
print('Done. Summary keys:', list(summary.keys()))

## Figure 1: Simulated Data with BASTION Estimates

Scatter of observed series with posterior mean Trend (black) and Trend+Seasonality (blue).  Red dashed lines mark injected outlier positions.

In [ ]:
n           = len(sim['y'])
t           = np.arange(1, n + 1)
y           = sim['y']
trend_mean  = np.asarray(summary['Trend_sum']['Mean']).ravel()
signal_mean = np.asarray(summary['Signal_sum']['Mean']).ravel()

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(t, y, s=8, color='black', zorder=1, label='Observed')
ax.plot(t, signal_mean, linewidth=1.5, color='blue',  zorder=3, label='Trend + Seasonality')
ax.plot(t, trend_mean,  linewidth=1.5, color='black', zorder=4, label='Trend')
for i in sim['outliers']:
    ax.axvline(i + 1, color='red', linestyle='--', linewidth=1.0, alpha=0.7)
ax.set_xlabel('Time', fontsize=11)
ax.set_ylabel('Value', fontsize=11)
ax.legend(fontsize=9, loc='upper right')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'figure1_simulated_data.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/figure1_simulated_data.pdf')

## Figure 2: Trend Comparison — TBATS / MSTL / STR / BASTION

4-panel comparison of trend estimates vs the true piecewise-linear trend. MSE computed against the true trend.  R reference `.rds` files are loaded from `BASTION/inst/extdata/` when available.

In [ ]:
T_true   = sim['T']
trend_lo = np.asarray(summary['Trend_sum']['CR_lower']).ravel()
trend_hi = np.asarray(summary['Trend_sum']['CR_upper']).ravel()

has_tbats = has_mstl = has_str = False
tbats_trend = mstl_trend = str_trend = None
str_trend_lo = str_trend_hi = None

try:
    import rdata, pandas as pd

    def _ts(obj, attrs):
        data, dim = np.array(obj), attrs.get('dim', None)
        cols = (attrs.get('dimnames') or [None, None])[1]
        if dim is not None and len(dim) == 2:
            data = data.reshape(dim, order='F')
            return pd.DataFrame(data, columns=list(cols) if cols else None)
        return pd.Series(data)

    _cd    = {**rdata.conversion.DEFAULT_CLASS_MAP, 'ts': _ts}
    _parse = lambda f: rdata.conversion.convert(
        rdata.parser.parse_file(os.path.join(EXTDATA_DIR, f)), constructor_dict=_cd)
    _praw  = lambda f: rdata.conversion.convert(
        rdata.parser.parse_file(os.path.join(EXTDATA_DIR, f)))

    tb = _parse('TBATScomp.rds')
    tbats_trend = np.asarray(tb['level']).ravel()
    has_tbats   = True

    ms = _parse('MSTL.rds')
    mstl_trend = np.asarray(ms['Trend']).ravel()
    has_mstl   = True

    sp = _praw('STR.rds')['output']['predictors']
    str_trend    = np.asarray(sp[0]['data']).ravel()
    str_trend_lo = np.asarray(sp[0]['lower']).ravel()
    str_trend_hi = np.asarray(sp[0]['upper']).ravel()
    has_str      = True
    print('R reference outputs loaded.')
except Exception as e:
    warnings.warn(f'R references unavailable ({e}). Comparison panels will be blank.')

def mse(est, truth):
    return float('nan') if est is None else float(np.mean((est - truth) ** 2))

n = len(T_true); t = np.arange(1, n + 1)
panels = [
    ('TBATS',   tbats_trend, None,         None,         has_tbats),
    ('MSTL',    mstl_trend,  None,         None,         has_mstl),
    ('STR',     str_trend,   str_trend_lo, str_trend_hi, has_str),
    ('BASTION', trend_mean,  trend_lo,     trend_hi,     True),
]

fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True, sharey=True)
for ax, (name, tr, lo, hi, ok) in zip(axes.flatten(), panels):
    ax.plot(t, T_true, linewidth=1.0, color='grey', linestyle='--', label='True trend')
    if ok and tr is not None:
        if lo is not None:
            ax.fill_between(t, lo, hi, color='grey', alpha=0.35)
        ax.plot(t, tr, linewidth=1.5, color='black', label='Estimated')
    score = f'MSE={mse(tr, T_true):.2f}' if (ok and tr is not None) else 'unavailable'
    ax.set_title(f'{name}  ({score})', fontsize=11)
    ax.set_xlabel('Time', fontsize=9)
    ax.set_ylabel('Value', fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'figure2_trend_comparison.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/figure2_trend_comparison.pdf')